# Masking-lite — Media-Pipe full-body tracking, masking, blurring & movement tracing

> **Lite copy for the masking day.** It lives in `thursday-masking/` and auto-locates the corpus, so you can run it from here without moving anything.
<br>
<div align="center">Wim Pouw (wim.pouw@donders.ru.nl) & Sho Akamine (Sho.Akamine@mpi.nl)</div>

<img src="../../Datasets/BalanceCorpus/scripts/motiontracking/Images/envision_banner.png" alt="isolated" width="300"/>
<img src="../../Datasets/BalanceCorpus/scripts/motiontracking/Images/options.gif" alt="isolated" width="600"/>

## Info documents

This python notebook runs you through the procedure of taking videos as inputs with a single person in the video, and outputting outputs of the kinematic timeseries, and optionally masking, blurring, and adding hand movement traces to videos with facial, hand, and arm skeletons.

The current code is derived and modified from the masked-piper tool, which is a simple but effective modification of the the Holistic Tracking by Google's Mediapipe so that we can use it as a CPU-based light weigth tool to mask your video data while maintaining background information, and also preserving information about body kinematics. 

* location Repository:  https://github.com/WimPouw/envisionBOX_modulesWP/tree/main/Mediapipe_Optional_Masking

* location Jupyter notebook: https://github.com/WimPouw/envisionBOX_modulesWP/blob/main/MultimodalMerging/Masking_Mediapiping.ipynb

Current Github: https://github.com/WimPouw/TowardsMultimodalOpenScience

## Version
2.0.0 (we added some functionalities such as blurring and movement tracing, and fixed some bugs with non-present right/left body parts)

## Additional information backbone of the tool (Mediapipe Holistic Tracking)
https://google.github.io/mediapipe/solutions/holistic.html

## Citation of mediapipe
citation: Lugaresi, C., Tang, J., Nash, H., McClanahan, C., Uboweja, E., Hays, M., ... & Grundmann, M. (2019). Mediapipe: A framework for building perception pipelines. arXiv preprint arXiv:1906.08172.

## Citation of masked piper
* citation: Owoyele, B., Trujillo, J., De Melo, G., & Pouw, W. (2022). Masked-Piper: Masking personal identities in visual recordings while preserving multimodal information. SoftwareX, 20, 101236. 
* Original Repo: https://github.com/WimPouw/TowardsMultimodalOpenScience

## Citation of this code
* citation: Pouw, W., & Akamine, S. (2025). Using Media-Pipe for full-body tracking, masking, blurring, and movement tracing. Retrieved from https://github.com/WimPouw/envisionBOX_modulesWP/tree/main/Mediapipe_Optional_Masking

## Use
Make sure to install all the packages in requirements.txt. Then move your videos that you want to mask into the input folder. Then run this code, which will loop through all the videos contained in the input folder; and saves all the results in the output folders.

## Setting up packages and folder structure

In [ ]:
# On Google Colab, install the video-masking libraries (skipped when running locally).
import sys
if "google.colab" in sys.modules:
    import subprocess
    subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                    "mediapipe==0.10.10", "opencv-python", "pandas", "numpy"], check=True)
    print("Colab: installed mediapipe + opencv.")
else:
    print("Local run - make sure the masking-video env is active (see HANDS_ON_SETUP.md).")

In [8]:
#load in required packages
import mediapipe as mp
import cv2
import math
import numpy as np
import pandas as pd
import csv
import os, sys

# --- where YOUR videos go, and where results are written ---
# Everything is kept NEXT TO THIS NOTEBOOK, so it is always writable (works on Colab too).
BASE = os.getcwd()
MYDIR = os.path.join(BASE, "my_videos")                       # drop your .mp4 files here
outputf_mask = os.path.join(BASE, "masking_output", "Output_Videos") + os.sep
outtputf_ts  = os.path.join(BASE, "masking_output", "Output_TimeSeries") + os.sep
for d in (MYDIR, outputf_mask, outtputf_ts):
    os.makedirs(d, exist_ok=True)

# optionally locate the bundled corpus (for a demo clip) -- fine if it is absent (e.g. on Colab)
def find_corpus(start=None):
    d = os.path.abspath(start or os.getcwd())
    for _ in range(10):
        cand = os.path.join(d, "Datasets", "BalanceCorpus")
        if os.path.isdir(os.path.join(cand, "videos")):
            return cand
        p = os.path.dirname(d)
        if p == d:
            break
        d = p
    return None
CORPUS = find_corpus()

print("Put your videos in :", MYDIR)
print("Results go to      :", os.path.abspath(os.path.join(BASE, "masking_output")))
print("Bundled corpus     :", CORPUS or "(not found - that is fine, just upload your own)")

The following folder is set as the output folder where all the pose time series are stored
d:\Research_projects\TilburgMultiscaleSummerschool2026\Datasets\BalanceCorpus\motiontracking\Output_TimeSeries

 The following folder is set as the output folder for saving the masked videos 
d:\Research_projects\TilburgMultiscaleSummerschool2026\Datasets\BalanceCorpus\motiontracking\Output_Videos

 The following video(s) will be processed for masking: 
['../../videos/103_203\\clue_giver\\103_203_12_1_20250113_152455_doughnut_board_clueGiver_cam01.mp4', '../../videos/103_203\\clue_giver\\103_203_12_1_20250113_152455_doughnut_board_clueGiver_cam02.mp4', '../../videos/103_203\\clue_giver\\103_203_13_1_20250113_152513_spinach_board_clueGiver_cam01.mp4', '../../videos/103_203\\clue_giver\\103_203_13_1_20250113_152513_spinach_board_clueGiver_cam02.mp4', '../../videos/103_203\\clue_giver\\103_203_14_1_20250113_152536_balloon_board_clueGiver_cam01.mp4', '../../videos/103_203\\clue_giver\\103_203_14_1_20

## Choose your video(s) — upload your own, or use the demo

Run the cell below. **On Colab** a file picker opens — pick one or more `.mp4` files (or Cancel to
use the demo). **Locally**, drop `.mp4` files into the `my_videos/` folder printed above and re-run.
If you provide nothing, it falls back to a single demo clip from the corpus.

In [ ]:
# Upload (Colab) or read local files, then decide which videos to process.
if "google.colab" in sys.modules:
    from google.colab import files
    print("Upload one or more .mp4 files (or press Cancel to use the demo):")
    try:
        uploaded = files.upload()
        for name, data in uploaded.items():
            with open(os.path.join(MYDIR, name), "wb") as fh:
                fh.write(data)
    except Exception as e:
        print("Upload skipped:", e)

def list_mp4(folder):
    return [os.path.join(dp, f) for dp, dn, fs in os.walk(folder)
            for f in fs if f.lower().endswith(".mp4") and not f.startswith("._")]

myvids = list_mp4(MYDIR)
if myvids:
    vfiles = myvids
    print(f"Using YOUR {len(vfiles)} video(s):")
elif CORPUS:
    vfiles = list_mp4(os.path.join(CORPUS, "videos"))[:1]   # one demo clip keeps this "lite"
    print(f"No uploads found -> using {len(vfiles)} demo clip from the corpus:")
else:
    vfiles = []
    print(f"No videos yet. Drop .mp4 files into: {MYDIR}")
for v in vfiles:
    print("  ", v)

## Initializing marker names and key methods from mediapipe

In [9]:
#initialize modules and functions

#load in mediapipe modules
mp_holistic = mp.solutions.holistic
# Import drawing_utils and drawing_styles.
mp_drawing = mp.solutions.drawing_utils
mp_drawing_styles = mp.solutions.drawing_styles

##################FUNCTIONS AND OTHER VARIABLES
#landmarks 33x that are used by Mediapipe (Blazepose)
markersbody = ['NOSE', 'LEFT_EYE_INNER', 'LEFT_EYE', 'LEFT_EYE_OUTER', 'RIGHT_EYE_OUTER', 'RIGHT_EYE', 'RIGHT_EYE_OUTER',
          'LEFT_EAR', 'RIGHT_EAR', 'MOUTH_LEFT', 'MOUTH_RIGHT', 'LEFT_SHOULDER', 'RIGHT_SHOULDER', 'LEFT_ELBOW', 
          'RIGHT_ELBOW', 'LEFT_WRIST', 'RIGHT_WRIST', 'LEFT_PINKY', 'RIGHT_PINKY', 'LEFT_INDEX', 'RIGHT_INDEX',
          'LEFT_THUMB', 'RIGHT_THUMB', 'LEFT_HIP', 'RIGHT_HIP', 'LEFT_KNEE', 'RIGHT_KNEE', 'LEFT_ANKLE', 'RIGHT_ANKLE',
          'LEFT_HEEL', 'RIGHT_HEEL', 'LEFT_FOOT_INDEX', 'RIGHT_FOOT_INDEX']

markershands = ['LEFT_WRIST', 'LEFT_THUMB_CMC', 'LEFT_THUMB_MCP', 'LEFT_THUMB_IP', 'LEFT_THUMB_TIP', 'LEFT_INDEX_FINGER_MCP',
              'LEFT_INDEX_FINGER_PIP', 'LEFT_INDEX_FINGER_DIP', 'LEFT_INDEX_FINGER_TIP', 'LEFT_MIDDLE_FINGER_MCP', 
               'LEFT_MIDDLE_FINGER_PIP', 'LEFT_MIDDLE_FINGER_DIP', 'LEFT_MIDDLE_FINGER_TIP', 'LEFT_RING_FINGER_MCP', 
               'LEFT_RING_FINGER_PIP', 'LEFT_RING_FINGER_DIP', 'LEFT_RING_FINGER_TIP', 'LEFT_PINKY_FINGER_MCP', 
               'LEFT_PINKY_FINGER_PIP', 'LEFT_PINKY_FINGER_DIP', 'LEFT_PINKY_FINGER_TIP',
              'RIGHT_WRIST', 'RIGHT_THUMB_CMC', 'RIGHT_THUMB_MCP', 'RIGHT_THUMB_IP', 'RIGHT_THUMB_TIP', 'RIGHT_INDEX_FINGER_MCP',
              'RIGHT_INDEX_FINGER_PIP', 'RIGHT_INDEX_FINGER_DIP', 'RIGHT_INDEX_FINGER_TIP', 'RIGHT_MIDDLE_FINGER_MCP', 
               'RIGHT_MIDDLE_FINGER_PIP', 'RIGHT_MIDDLE_FINGER_DIP', 'RIGHT_MIDDLE_FINGER_TIP', 'RIGHT_RING_FINGER_MCP', 
               'RIGHT_RING_FINGER_PIP', 'RIGHT_RING_FINGER_DIP', 'RIGHT_RING_FINGER_TIP', 'RIGHT_PINKY_FINGER_MCP', 
               'RIGHT_PINKY_FINGER_PIP', 'RIGHT_PINKY_FINGER_DIP', 'RIGHT_PINKY_FINGER_TIP']
facemarks = [str(x) for x in range(478)] #there are 478 points for the face mesh (see google holistic face mesh info for landmarks)

print("Note that we have the following number of pose keypoints for markers body")
print(len(markersbody))

print("\n Note that we have the following number of pose keypoints for markers hands")
print(len(markershands))

print("\n Note that we have the following number of pose keypoints for markers face")
print(len(facemarks ))

#set up the column names and objects for the time series data (add time as the first variable)
markerxyzbody = ['time']
markerxyzhands = ['time']
markerxyzface = ['time']

for mark in markersbody:
    for pos in ['X', 'Y', 'Z', 'visibility']: #for markers of the body you also have a visibility reliability score
        nm = pos + "_" + mark
        markerxyzbody.append(nm)
for mark in markershands:
    for pos in ['X', 'Y', 'Z']:
        nm = pos + "_" + mark
        markerxyzhands.append(nm)
for mark in facemarks:
    for pos in ['X', 'Y', 'Z']:
        nm = pos + "_" + mark
        markerxyzface.append(nm)

#check if there are numbers in a string
def num_there(s):
    return any(i.isdigit() for i in s)

#take some google classification object and convert it into a string
def makegoginto_str(gogobj):
    gogobj = str(gogobj).strip("[]")
    gogobj = gogobj.split("\n")
    return(gogobj[:-1]) #ignore last element as this has nothing

#make the stringifyd position traces into clean numerical values
def listpostions(newsamplemarks):
    newsamplemarks = makegoginto_str(newsamplemarks)
    tracking_p = []
    for value in newsamplemarks:
        if num_there(value):
            stripped = value.split(':', 1)[1]
            stripped = stripped.strip() #remove spaces in the string if present
            tracking_p.append(stripped) #add to this list  
    return(tracking_p)

Note that we have the following number of pose keypoints for markers body
33

 Note that we have the following number of pose keypoints for markers hands
42

 Note that we have the following number of pose keypoints for markers face
478


## Main procedure — broken into steps

The original tool did everything in one long loop. Here it is the **same pipeline**, but each
per-frame operation is pulled out into its own small, named function so you can read, teach, and
tweak them one at a time. Nothing about the behaviour changes — the final loop (Step 7) calls these
in exactly the original order, under the same on/off switches below.

Pipeline per frame: **choose background → (body mask) → (body blur) → (face blur) → (face mask) →
finger traces → draw skeleton → record kinematics**.

### Options — the on/off switches

These flags are the dials you play with. The default is *full-body blur + full skeleton + index-finger
traces*. Every function in the steps below reads these.

In [ ]:
# MASKING AND BLURRING OPTIONS
skeleton = True
skeleton_face_only = False         # Only show face skeleton (no body, no hands)
whitebackground = False            # Grey background with skeleton only (no body, no face)
maskingbody = False                # Masks the body silhouette with fully black color (original masked-piper approach)
maskingface = False                # Masks the face (fully black color)
blurringface = False               # Blurs the face
blurringbody = True                # Blurs the body region
blurringfactor = 1                 # Blurring intensity (0-1, 1 is full blur)
showrender = False                 # Show the rendered video in a window while processing
# TRACE OPTIONS
add_finger_traces = True           # Add fading traces for index fingers
trace_length_seconds = 1.5         # Length of trace in seconds
trace_color_left = (0, 255, 0)     # Green for left index finger trace
trace_color_right = (0, 0, 255)    # Blue for right index finger trace

### Step 1 · Choose the background

Every frame starts from one of two canvases: the **white background** mode (skeleton floats on
white, body/face removed) or the **original frame** (we then mask/blur on top of it).

In [ ]:
def choose_background(image, results, h, w, whitebackground):
    """Return the starting BGR canvas for this frame."""
    if whitebackground:
        white_image = np.full((h, w, 3), (255, 255, 255), dtype=np.uint8)
        return cv2.cvtColor(white_image, cv2.COLOR_RGB2BGR)
    return cv2.cvtColor(image, cv2.COLOR_RGB2BGR)

### Step 2 · Mask or blur the *body*

`mask_body` is the original Masked-Piper move: use MediaPipe's segmentation mask to black out the
body silhouette while keeping the background. `blur_body` is the gentler alternative: Gaussian-blur
only the body region so shape/motion survive but identity is degraded.

In [ ]:
def mask_body(image, results, h, w):
    """Original masked-piper: black out the body silhouette using the segmentation mask."""
    image_with_alpha = np.concatenate([image, np.full((h, w, 1), 255, dtype=np.uint8)], axis=-1)
    mask_img = np.zeros_like(image, dtype=np.uint8)
    mask_img[:, :] = (255, 255, 255)
    segm_2class = 0.2 + 0.8 * results.segmentation_mask
    segm_2class = np.repeat(segm_2class[..., np.newaxis], 3, axis=2)
    annotated_image = mask_img * segm_2class * (1 - segm_2class)
    mask = np.concatenate([annotated_image, np.full((h, w, 1), 255, dtype=np.uint8)], axis=-1)
    image_with_alpha[mask == 0] = 0
    return cv2.cvtColor(image_with_alpha, cv2.COLOR_RGB2BGR)

def blur_body(original_image, results, blurringfactor):
    """Gaussian-blur only the body region (identity down, shape/motion kept)."""
    segm_2class = 0.2 + 0.8 * results.segmentation_mask
    body_mask = segm_2class
    kernel_size = int(51 * blurringfactor)
    if kernel_size % 2 == 0:
        kernel_size += 1
    blurred_image = cv2.GaussianBlur(original_image, (kernel_size, kernel_size), 0)
    body_mask_3channel = cv2.merge([body_mask] * 3)
    return (original_image * (1 - body_mask_3channel * blurringfactor) +
            blurred_image * (body_mask_3channel * blurringfactor)).astype(np.uint8)

### Step 3 · Blur or mask the *face*

Same idea, but scoped to the face via a convex hull over the 478 face-mesh points. `blur_face`
softens it; `mask_face` blacks it out entirely.

In [ ]:
def blur_face(original_image, results, h, w, blurringfactor):
    """Gaussian-blur only the face region (convex hull over the face mesh)."""
    face_mask = np.zeros((h, w), dtype=np.uint8)
    landmarks = results.face_landmarks.landmark
    face_points = np.array([(int(landmarks[i].x * w), int(landmarks[i].y * h))
                            for i in range(len(landmarks))], dtype=np.int32)
    hull = cv2.convexHull(face_points)
    cv2.fillConvexPoly(face_mask, hull, 255)
    kernel_size = int(51 * blurringfactor)
    if kernel_size % 2 == 0:
        kernel_size += 1
    blurred_image = cv2.GaussianBlur(original_image, (kernel_size, kernel_size), 0)
    face_mask_3channel = cv2.cvtColor(face_mask, cv2.COLOR_GRAY2BGR) / 255.0
    return (original_image * (1 - face_mask_3channel * blurringfactor) +
            blurred_image * (face_mask_3channel * blurringfactor)).astype(np.uint8)

def mask_face(original_image, results, h, w):
    """Make the face fully black (convex hull over the face mesh)."""
    face_mask = np.zeros((h, w), dtype=np.uint8)
    landmarks = results.face_landmarks.landmark
    face_points = np.array([(int(landmarks[i].x * w), int(landmarks[i].y * h))
                            for i in range(len(landmarks))], dtype=np.int32)
    hull = cv2.convexHull(face_points)
    cv2.fillConvexPoly(face_mask, hull, 255)
    original_image[face_mask > 0] = (0, 0, 0)
    return original_image

### Step 4 · Fading finger-motion traces

Optionally draw a fading tail behind each index-finger tip (landmark 8) — a lightweight way to *show*
gesture motion on the masked video. The trace buffers persist across frames within a video, capped to
`trace_length_seconds`.

In [ ]:
def update_finger_traces(original_image, results, left_finger_trace, right_finger_trace,
                         trace_frames, w, h, trace_color_left, trace_color_right):
    """Append current index-finger tips to the trace buffers and draw fading tails."""
    left_finger_pos = None
    right_finger_pos = None
    if results.left_hand_landmarks:
        left_landmark = results.left_hand_landmarks.landmark[8]   # index-finger tip
        left_finger_pos = (int(left_landmark.x * w), int(left_landmark.y * h))
    if results.right_hand_landmarks:
        right_landmark = results.right_hand_landmarks.landmark[8]
        right_finger_pos = (int(right_landmark.x * w), int(right_landmark.y * h))
    if left_finger_pos:
        left_finger_trace.append(left_finger_pos)
    if right_finger_pos:
        right_finger_trace.append(right_finger_pos)
    if len(left_finger_trace) > trace_frames:
        left_finger_trace.pop(0)
    if len(right_finger_trace) > trace_frames:
        right_finger_trace.pop(0)
    for i in range(len(left_finger_trace) - 1):
        if i < len(left_finger_trace) and (i + 1) < len(left_finger_trace):
            opacity = int(255 * (i + 1) / len(left_finger_trace)); alpha = opacity / 255.0
            overlay = original_image.copy()
            cv2.line(overlay, left_finger_trace[i], left_finger_trace[i + 1], trace_color_left, 2)
            original_image = cv2.addWeighted(overlay, alpha, original_image, 1 - alpha, 0)
    for i in range(len(right_finger_trace) - 1):
        if i < len(right_finger_trace) and (i + 1) < len(right_finger_trace):
            opacity = int(255 * (i + 1) / len(right_finger_trace)); alpha = opacity / 255.0
            overlay = original_image.copy()
            cv2.line(overlay, right_finger_trace[i], right_finger_trace[i + 1], trace_color_right, 2)
            original_image = cv2.addWeighted(overlay, alpha, original_image, 1 - alpha, 0)
    return original_image

### Step 5 · Draw the skeleton

Overlay the MediaPipe landmarks — hands, face mesh, and body pose (or face-only). This is the
keypoint layer MaskBench and the gesture analyses read.

In [ ]:
def draw_skeleton(original_image, results, skeleton, skeleton_face_only):
    """Overlay hand / face-mesh / pose landmarks in place."""
    if skeleton:
        mp_drawing.draw_landmarks(original_image, results.left_hand_landmarks, mp_holistic.HAND_CONNECTIONS)
        mp_drawing.draw_landmarks(original_image, results.right_hand_landmarks, mp_holistic.HAND_CONNECTIONS)
        mp_drawing.draw_landmarks(
            original_image,
            results.face_landmarks,
            mp_holistic.FACEMESH_TESSELATION,
            landmark_drawing_spec=None,
            connection_drawing_spec=mp_drawing_styles.get_default_face_mesh_tesselation_style()
        )
        mp_drawing.draw_landmarks(
            original_image,
            results.pose_landmarks,
            mp_holistic.POSE_CONNECTIONS,
            landmark_drawing_spec=mp_drawing_styles.get_default_pose_landmarks_style()
        )
    elif skeleton_face_only:
        mp_drawing.draw_landmarks(
            original_image,
            results.face_landmarks,
            mp_holistic.FACEMESH_TESSELATION,
            landmark_drawing_spec=None,
            connection_drawing_spec=mp_drawing_styles.get_default_face_mesh_tesselation_style()
        )

### Step 6 · Record the kinematic time series

Pull the numeric landmark positions for this frame (body, hands, face) and prepend the timestamp.
The hand handling keeps left/right from swapping when one hand is missing.

In [ ]:
def extract_timeseries_rows(results, time, markerxyzhands):
    """Return (body, hands, face) rows for this frame, each starting with `time`."""
    samplebody = listpostions(results.pose_landmarks)
    sampleface = listpostions(results.face_landmarks)
    # Process hands separately (fixes the left/right hand swapping issue)
    sampleLH = listpostions(results.left_hand_landmarks)
    sampleRH = listpostions(results.right_hand_landmarks)
    if len(sampleLH) == 0:
        sampleLH = ["" for x in range(int(len(markerxyzhands) / 2))]
    samplehands = sampleLH + sampleRH
    samplebody.insert(0, time)
    samplehands.insert(0, time)
    sampleface.insert(0, time)
    return samplebody, samplehands, sampleface

### Step 7 · Run the pipeline over every video

The driver. For each video it opens a writer, then per frame runs Holistic and calls the steps above
**in the original order and under the same switches**, writes the annotated frame, and finally saves
the body / hands / face CSVs. (Identical behaviour to the original single-cell version.)

In [ ]:
# Process videos
for vidf in vfiles:
    print(f"Processing video: {vidf}")
    print(f"Video {vfiles.index(vidf)+1} of {len(vfiles)}")

    videoname = vidf
    videoloc = videoname
    capture = cv2.VideoCapture(videoloc)
    frameWidth = capture.get(cv2.CAP_PROP_FRAME_WIDTH)
    frameHeight = capture.get(cv2.CAP_PROP_FRAME_HEIGHT)
    samplerate = capture.get(cv2.CAP_PROP_FPS)

    # check if file already exist, otherwise skip
    if os.path.exists(outputf_mask + os.path.basename(vidf)):
        print(f"Output video {outputf_mask + os.path.basename(vidf)} already exists. Skipping processing for this video.")
        continue
    fourcc = cv2.VideoWriter_fourcc(*'MP4V')
    out = cv2.VideoWriter(outputf_mask + os.path.basename(vidf), fourcc,
                          fps=samplerate, frameSize=(int(frameWidth), int(frameHeight)))

    time = 0
    tsbody = [markerxyzbody]
    tshands = [markerxyzhands]
    tsface = [markerxyzface]

    # Initialize trace buffers for index fingers
    if add_finger_traces:
        trace_frames = int(trace_length_seconds * samplerate)
        left_finger_trace = []   # (x, y) positions for left index finger
        right_finger_trace = []  # (x, y) positions for right index finger

    with mp_holistic.Holistic(static_image_mode=False, enable_segmentation=True, refine_face_landmarks=True) as holistic:
        while True:
            ret, image = capture.read()
            if ret == True:
                image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
                results = holistic.process(image)
                h, w, c = image.shape

                if np.all(results.face_landmarks) != None:
                    # Step 1: background (white mode overrides the others)
                    original_image = choose_background(image, results, h, w, whitebackground)
                    # Step 2: body mask / blur
                    if maskingbody and not whitebackground:
                        original_image = mask_body(image, results, h, w)
                    if blurringbody and not whitebackground:
                        original_image = blur_body(original_image, results, blurringfactor)
                    # Step 3: face blur / mask
                    if blurringface and results.face_landmarks and not whitebackground:
                        original_image = blur_face(original_image, results, h, w, blurringfactor)
                    if maskingface and results.face_landmarks and not whitebackground:
                        original_image = mask_face(original_image, results, h, w)
                    # Step 4: finger traces
                    if add_finger_traces:
                        original_image = update_finger_traces(
                            original_image, results, left_finger_trace, right_finger_trace,
                            trace_frames, w, h, trace_color_left, trace_color_right)
                    # Step 5: skeleton
                    draw_skeleton(original_image, results, skeleton, skeleton_face_only)
                    # Step 6: kinematics
                    samplebody, samplehands, sampleface = extract_timeseries_rows(results, time, markerxyzhands)
                    tsbody.append(samplebody)
                    tshands.append(samplehands)
                    tsface.append(sampleface)
                else:
                    original_image = cv2.cvtColor(image, cv2.COLOR_RGB2BGR)
                    # Add NaN data
                    samplebody = [np.nan for x in range(len(markerxyzbody)-1)]
                    samplehands = [np.nan for x in range(len(markerxyzhands)-1)]
                    sampleface = [np.nan for x in range(len(markerxyzface)-1)]
                    samplebody.insert(0, time)
                    samplehands.insert(0, time)
                    sampleface.insert(0, time)
                    tsbody.append(samplebody)
                    tshands.append(samplehands)
                    tsface.append(sampleface)

                if showrender:
                    cv2.imshow("resizedimage", original_image)
                out.write(original_image)
                time = time + (1000/samplerate)

            if cv2.waitKey(1) == 27:
                break
            if ret == False:
                break

    out.release()
    capture.release()
    cv2.destroyAllWindows()

    # Save CSV files
    filebody = open(outtputf_ts + os.path.basename(vidf)[:-4] + '_body.csv', 'w+', newline='')
    with filebody:
        write = csv.writer(filebody)
        write.writerows(tsbody)

    filehands = open(outtputf_ts + os.path.basename(vidf)[:-4] + '_hands.csv', 'w+', newline='')
    with filehands:
        write = csv.writer(filehands)
        write.writerows(tshands)

    fileface = open(outtputf_ts + os.path.basename(vidf)[:-4] + '_face.csv', 'w+', newline='')
    with fileface:
        write = csv.writer(fileface)
        write.writerows(tsface)

print("Done with processing all folders; go look in your output folders!")

## Example output (full body blur + skeleton + Tracing)
<img src="../../Datasets/BalanceCorpus/scripts/motiontracking/Images/ted_kid.gif" alt="isolated" width="600"/>